# Unrevised-Data Check for Market Macro Series

This notebook checks whether the market-observed FRED series used as contemporaneous predictors appear unrevised over the sample period.

Main test for each series:

$$\text{revision}_t = x_t^{\text{current FRED}} - x_t^{\text{ALFRED first release}}$$

A series passes the unrevised-value check when comparable observations have zero changed values and zero maximum absolute revision. Publication timing is checked separately using `realtime_start - date` from the ALFRED first-release records.

`NBUSBIS` needs one extra unit-harmonisation step before revision testing. Older first-release vintages were published as `Index 2010=100`, while current FRED history is on `Index 2020=100`. The notebook rebases pre-2023 first-release vintages to `Index 2020=100` before comparing them with current FRED values.

Series checked: `T10Y2Y`, `DGS3MO`, `BAA10Y`, `T5YIE`, `T10YIE`, `VIXCLS`, `DCOILWTICO`, `NBUSBIS`.

In [ ]:
import datetime as dt
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

FRED_OBSERVATIONS_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_SERIES_URL = "https://api.stlouisfed.org/fred/series"

START = "2000-01-01"
END = "2026-07-31"
SERIES_IDS = [
    "T10Y2Y",
    "DGS3MO",
    "BAA10Y",
    "T5YIE",
    "T10YIE",
    "VIXCLS",
    "DCOILWTICO",
    "NBUSBIS",
]

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "macro_variables":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif (PROJECT_ROOT / "dataset" / "macro_variables").exists():
    PROJECT_ROOT = PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd().resolve()


print(f"Project root: {PROJECT_ROOT}")


## API Helpers

`output_type=4` returns initial-release observations. The ALFRED call is chunked by real-time year because long daily-series vintage windows can exceed the FRED JSON API vintage-date limit.

In [ ]:
def load_dotenv_fallback(path=None):
    """Minimal .env loader so FRED_API_KEY can live in the repo-level .env file."""
    if path is None:
        path = PROJECT_ROOT / ".env"
    path = Path(path)
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_fred_api_key():
    load_dotenv_fallback()
    api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise RuntimeError("Missing FRED_API_KEY. Set it in your shell or repo-level .env before running this notebook.")
    return api_key


def verify_series_exists(series_id, api_key):
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
    }
    response = requests.get(FRED_SERIES_URL, params=params, timeout=30)
    try:
        payload = response.json()
    except ValueError:
        response.raise_for_status()
        raise
    if response.status_code >= 400 or "error_message" in payload:
        raise RuntimeError(
            f"{series_id} does not exist in current FRED: "
            f"{payload.get('error_message', response.text)}"
        )
    return True


def parse_observations(payload):
    df = pd.DataFrame(payload.get("observations", []))
    if df.empty:
        return pd.DataFrame(columns=["date", "realtime_start", "realtime_end", "value"])
    df["date"] = pd.to_datetime(df["date"])
    if "realtime_start" in df.columns:
        df["realtime_start"] = pd.to_datetime(df["realtime_start"])
    if "realtime_end" in df.columns:
        df["realtime_end"] = pd.to_datetime(df["realtime_end"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df


def fred_get(params, timeout=30):
    response = requests.get(FRED_OBSERVATIONS_URL, params=params, timeout=timeout)
    try:
        payload = response.json()
    except ValueError:
        response.raise_for_status()
        raise
    if response.status_code >= 400 or "error_message" in payload:
        message = payload.get("error_message", response.text)
        raise RuntimeError(message)
    return payload


def is_empty_vintage_window_error(exc):
    message = str(exc).lower()
    return (
        "no vintage dates exist" in message
        or "no observations" in message
        or "series does not exist" in message
    )


def yearly_chunks(start, end):
    start_dt = pd.Timestamp(start)
    end_dt = pd.Timestamp(end)
    chunk_start = start_dt
    while chunk_start <= end_dt:
        chunk_end = min(pd.Timestamp(year=chunk_start.year, month=12, day=31), end_dt)
        yield chunk_start.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")
        chunk_start = chunk_end + pd.Timedelta(days=1)


def fetch_alfred_first_release(series_id, api_key, start=START, end=END):
    # First prove the series ID is valid in current FRED. After this check,
    # historical ALFRED "series does not exist" errors mean unavailable in that vintage window.
    verify_series_exists(series_id, api_key)

    frames = []
    for rt_start, rt_end in yearly_chunks(start, end):
        params = {
            "api_key": api_key,
            "file_type": "json",
            "series_id": series_id,
            "output_type": 4,
            "realtime_start": rt_start,
            "realtime_end": rt_end,
            "observation_start": start,
            "observation_end": end,
            "sort_order": "asc",
        }
        try:
            payload = fred_get(params)
        except RuntimeError as exc:
            if is_empty_vintage_window_error(exc):
                continue
            raise
        frame = parse_observations(payload)
        if not frame.empty:
            frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["date", "realtime_start", "realtime_end", "value"])
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["date"]).sort_values(["date", "realtime_start"])
    # Chunk boundaries can duplicate records; keep one first-release row per observation date.
    df = df.drop_duplicates(subset=["date"], keep="first")
    return df.reset_index(drop=True)


def fetch_current_fred(series_id, api_key, start=START, end=END):
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
        "observation_start": start,
        "observation_end": end,
        "sort_order": "asc",
    }
    payload = fred_get(params)
    return parse_observations(payload).sort_values("date").reset_index(drop=True)

## Comparison Functions

In [ ]:
def harmonize_units_for_revision_test(series_id, compare):
    compare = compare.copy()
    compare["first_release_value_raw"] = compare["first_release_value"]
    compare["current_value_raw"] = compare["current_value"]
    compare["unit_adjustment"] = "none"

    if series_id != "NBUSBIS":
        return compare

    unit_change_date = pd.Timestamp("2023-01-01")
    old_base_mask = compare["realtime_start"] < unit_change_date
    base_2020_mask = old_base_mask & (compare["date"].dt.year == 2020)
    old_base_2020_mean = compare.loc[base_2020_mask, "first_release_value"].mean()

    if pd.isna(old_base_2020_mean) or old_base_2020_mean == 0:
        raise RuntimeError("Cannot rebase NBUSBIS: missing nonzero 2020 old-base first-release values.")

    rebase_factor = 100.0 / old_base_2020_mean
    compare.loc[old_base_mask, "first_release_value"] = (
        compare.loc[old_base_mask, "first_release_value"] * rebase_factor
    )
    compare.loc[old_base_mask, "unit_adjustment"] = (
        f"rebased from Index 2010=100 to Index 2020=100; factor={rebase_factor:.10f}"
    )
    compare.loc[~old_base_mask, "unit_adjustment"] = "already Index 2020=100"
    return compare


def compare_first_release_to_current(series_id, api_key, tolerance=0.0):
    first = fetch_alfred_first_release(series_id, api_key)
    current = fetch_current_fred(series_id, api_key)

    first = first.rename(columns={"value": "first_release_value"})
    current = current.rename(columns={"value": "current_value"})[["date", "current_value"]]

    compare = first[["date", "realtime_start", "first_release_value"]].merge(
        current, on="date", how="inner"
    )
    compare = compare.dropna(subset=["first_release_value", "current_value"]).copy()
    compare = harmonize_units_for_revision_test(series_id, compare)
    compare["revision"] = compare["current_value"] - compare["first_release_value"]
    compare["abs_revision"] = compare["revision"].abs()
    compare["changed"] = compare["abs_revision"] > tolerance

    compare["publication_lag_days"] = (compare["realtime_start"] - compare["date"]).dt.days
    compare["publication_lag_present"] = compare["publication_lag_days"] > 0

    summary = {
        "series": series_id,
        "n_observations_checked": int(len(compare)),
        "n_revised_observations": int(compare["changed"].sum()),
        "max_abs_revision": float(compare["abs_revision"].max()) if len(compare) else np.nan,
        "mean_abs_revision": float(compare["abs_revision"].mean()) if len(compare) else np.nan,
        "publication_lag_present": bool(compare["publication_lag_present"].any()) if len(compare) else pd.NA,
        "max_publication_lag_days": float(compare["publication_lag_days"].max()) if len(compare) else np.nan,
        "median_publication_lag_days": float(compare["publication_lag_days"].median()) if len(compare) else np.nan,
        "first_comparable_date": compare["date"].min() if len(compare) else pd.NaT,
        "last_comparable_date": compare["date"].max() if len(compare) else pd.NaT,
        "treatment": "Current FRED if no revisions detected; otherwise revisit source/alignment",
        "status": "OK" if len(compare) else "NO_COMPARABLE_OBSERVATIONS",
    }
    return summary, compare


def run_unrevised_check(series_ids=SERIES_IDS):
    api_key = get_fred_api_key()
    summaries = []
    comparisons = {}

    for series_id in series_ids:
        print(f"Checking {series_id} ...")
        try:
            summary, compare = compare_first_release_to_current(series_id, api_key)
            comparisons[series_id] = compare
        except Exception as exc:
            summary = {
                "series": series_id,
                "n_observations_checked": 0,
                "n_revised_observations": pd.NA,
                "max_abs_revision": np.nan,
                "mean_abs_revision": np.nan,
                "publication_lag_present": pd.NA,
                "max_publication_lag_days": np.nan,
                "median_publication_lag_days": np.nan,
                "first_comparable_date": pd.NaT,
                "last_comparable_date": pd.NaT,
                "treatment": "Audit failed; inspect API/source availability",
                "status": f"ERROR: {exc}",
            }
            comparisons[series_id] = pd.DataFrame()
        summaries.append(summary)

    summary_df = pd.DataFrame(summaries)
    return summary_df, comparisons

## Run Check

This cell displays the summary table and keeps detailed per-series comparisons in memory only.

In [ ]:
summary_df, comparisons = run_unrevised_check()
summary_df

## Pass/Fail View

The ideal unrevised-data result is `n_revised_observations == 0` and `max_abs_revision == 0`. Publication lag is interpreted separately from value revisions.

In [ ]:
pass_fail = summary_df.copy()
pass_fail["passes_unrevised_value_test"] = (
    (pass_fail["status"] == "OK")
    & (pass_fail["n_revised_observations"] == 0)
    & (pass_fail["max_abs_revision"].fillna(np.inf) == 0)
)
pass_fail[[
    "series",
    "n_observations_checked",
    "n_revised_observations",
    "max_abs_revision",
    "mean_abs_revision",
    "publication_lag_present",
    "max_publication_lag_days",
    "treatment",
    "passes_unrevised_value_test",
    "status",
]]

## Largest Revisions, If Any

In [ ]:
for series_id, compare in comparisons.items():
    if compare.empty or not compare["changed"].any():
        continue
    print(f"\n{series_id}: largest absolute revisions")
    display(
        compare.sort_values("abs_revision", ascending=False)[[
            "date",
            "realtime_start",
            "first_release_value",
            "current_value",
            "revision",
            "abs_revision",
            "publication_lag_days",
        ]].head(20)
    )

## Methodology Wording

For the daily FRED series `T10Y2Y`, `DGS3MO`, `BAA10Y`, `T5YIE`, `T10YIE`, `VIXCLS`, `DCOILWTICO` and `NBUSBIS`, we compared each observation's ALFRED initial-release value with its current FRED value over the sample period. A series was treated as unrevised only when no comparable historical observation changed across vintages. Publication timing was assessed separately to ensure that observation dates did not introduce look-ahead bias.